## IMPORTS

In [ ]:
import csv
from datetime import datetime
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import tifffile
import torch
from codecarbon import EmissionsTracker
from ultralytics import SAM

## PARAMETRES

In [ ]:
GRID_STRIDE = 64  # densité des points
POINTS_PER_CALL = 30  # taille des chunks envoyés à SAM
MIN_AREA = 300  # aire min d’un masque (en pixels)
CONF_THR = 0.35  # seuil min confiance de SAM
DEDUP_IOU_THR = 0.90  # seuil de dé-duplication
MIN_MASK_REGION_AREA = 200
IMAGE_3D_PATH = "../data/Romane_Martin_urne_sature_10-4.tif"
SAM_WEIGHTS = "../data/sam_b.pt"  # path sam en local ou bien download from ultralytics
POINT_LABEL = 1

## UTILS

In [ ]:
def make_grid_points(h, w, stride, label=1):
    xs = np.arange(stride // 2, w, stride)
    ys = np.arange(stride // 2, h, stride)
    pts = [(int(x), int(y)) for y in ys for x in xs]  # (x, y)
    lbls = [label] * len(pts)
    return pts, lbls


def dedup_by_iou(masks, iou_thr=0.9, min_area=0):
    keep = []
    for m in masks:
        if m.sum() < min_area:
            continue
        if any(
            (np.logical_and(m, k).sum() / max(np.logical_or(m, k).sum(), 1)) > iou_thr
            for k in keep
        ):
            continue
        keep.append(m)
    return keep


def colorize_masks(image_gray, masks_bool, seed=42):

    out = np.dstack([image_gray, image_gray, image_gray]).copy()  # [H, W, 3]
    out = out.astype(np.uint8, copy=False)
    rng = np.random.default_rng(seed)
    for m in masks_bool:
        out[m] = rng.integers(0, 256, size=3, dtype=np.uint8)  # couleur aléatoire (R,G,B)
    return out

## CHARGEMENT DONNEES & MODELE


## chargement données

In [ ]:
def to_sam_handled_picture(picture_3D: np.ndarray) -> np.ndarray:
    return np.repeat(picture_3D[..., np.newaxis], 3, -1)

In [ ]:
# exemple extraction slice au milieu
vol = tifffile.imread(IMAGE_3D_PATH)
print(f"Volume: shape={vol.shape}, dtype={vol.dtype}")
mid = len(vol) // 2
sl = vol[mid]

In [ ]:
plt.imshow(sl, cmap="gray")
plt.show()

## chargement SAM

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

if not Path(SAM_WEIGHTS).exists():
    print(f"Downloading {SAM_WEIGHTS}...")
    model = SAM(SAM_WEIGHTS)
else:
    print(f"{SAM_WEIGHTS} already exists locally")
    model = SAM(SAM_WEIGHTS)

print("SAM chargé")

## INFERENCE PAR CHUNKS DE POINTS


In [ ]:
from typing import Union


def run_sam_chunked_points(
    model: SAM,
    image,
    points: Union[list[tuple[int, int]], int],
    labels=None,
    conf=0.35,
    points_per_call=25,
    device="cpu",
    binarize_thr=0.5,
):
    img = image if image.ndim == 3 else np.dstack([image, image, image])
    # >>>> ???? >>>>
    if img.dtype != np.uint8:
        mn, mx = float(img.min()), float(img.max())
        img = (
            ((img - mn) / (mx - mn) * 255).astype(np.uint8)
            if mx > mn
            else np.zeros_like(img, dtype=np.uint8)
        )
    img = np.ascontiguousarray(img)
    # <<<< ???? <<<<
    masks = []
    with torch.inference_mode():
        if isinstance(points, int):
            r = model.predict(
                source=img,
                points_stride=points,
                points_batch_size=points_per_call,
                conf=conf,
                device=device,
            )
            if getattr(r, "masks", None) is not None and r.masks is not None:
                t = r.masks.data  # [N,H,W]
                arr = (t > binarize_thr).cpu().numpy()
                masks.extend(mi.astype(bool) for mi in arr)
        else:
            for i in range(0, len(points), points_per_call):
                r = model.predict(
                    source=img,
                    points=points[i : i + points_per_call],
                    labels=labels[i : i + points_per_call],
                    conf=conf,
                    device=device,
                )[0]
                if getattr(r, "masks", None) is not None and r.masks is not None:
                    t = r.masks.data  # [N,H,W]
                    arr = (t > binarize_thr).cpu().numpy()
                    masks.extend(mi.astype(bool) for mi in arr)
        if device == "cuda":
            torch.cuda.empty_cache()
    return masks  # non dédupliqués

### grille de points et inférence

In [ ]:
type(sl)

In [ ]:
tracker = EmissionsTracker()
tracker.start()
H, W = sl.shape[:2]
points, labels = make_grid_points(H, W, GRID_STRIDE, label=POINT_LABEL)
print(f"Points de grille: {len(points)}  (stride={GRID_STRIDE})")

In [ ]:
raw_masks = run_sam_chunked_points(
    model, sl, points, labels, conf=CONF_THR, points_per_call=POINTS_PER_CALL, device=device
)

In [ ]:
masks = dedup_by_iou(raw_masks, iou_thr=DEDUP_IOU_THR, min_area=MIN_AREA)
print(f" {len(masks)} masques après filtrage & dé-dup")

## SAVE VISUALISATION

In [ ]:
if len(masks) == 0:
    print(" Aucun objet détecté ")
else:
    masks_sorted = sorted(masks, key=lambda x: x.sum(), reverse=True)
    colored = colorize_masks(sl, masks_sorted, seed=0)
    plt.imshow(colored)
    plt.axis("off")
    plt.show()
    cv2.imwrite(f"sam_colored_slice_{mid}.png", colored)
    print(f" Sauvegarde: sam_colored_slice_{mid}.png")

### WRITE CODECARBONE RESULTS AND STOP TRACKER

In [ ]:
emissions = tracker.stop()
with open("emissions.csv", "a", newline="") as f:
    writer = csv.writer(f)
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    writer.writerow([timestamp, emissions])